In [1]:
from transformers import pipeline
import pandas as pd

In [5]:
jrue = """
Long and wiry combo guard \u2026 A crafty ballhandler that knows how to get defenders off balance with the dribble \u2026 Has a good repertoire of moves and mixes them up well to keep the defense guessing \u2026 He has a deceptive 1st step and shifty quickness making him difficult to contain on the perimeter \u2026 Is adept using both hands, either when attacking or finishing \u2026 Uses his body as well as his length to finish around the basket \u2026 Utilizes a variety of floaters and runners in the lane \u2026 Likes to pull up from the midrange where he shoots with balance and good rhythm \u2026 Shows good speed in the open court and the ability to manoeuvre through traffic with the dribble while going full speed  \u2026 Puts in a good effort defensively, where he enjoys being aggressive and pressuring ballhandlers \u2026 Has good hands, anticipates well and uses his wingspan to get many deflections \u2026 He is unselfish and possesses good court vision and has shown glimpses of being able to run a team full time \u2026"], "descr_weaknesses": ["Had a very disappointing season in terms of the hype he had coming in from highschool \u2026 Played out of position for the majority of the season, and struggled finding his comfort zone \u2026 Battled inconsistency with his shot all year and it threw the rest of his offensive game off balance \u2026 Defenses showed very little respect for his outside shot, daring him to shoot and taking away his driving lanes \u2026 His form is a big issue as it throws his stroke off and makes his release inconsistent \u2026 He shoots off the side of his head with the shooting elbow way out, as a result his shot is all over the place \u2026He does not have the superb athleticism or strength like many other combo guards \u2026 Tends to waste dribbles on the perimeter, killing the flow of the offense because he can be a ball stopper at times \u2026 Gets into trouble by over penetrating and then trying to make the spectacular play while in traffic and under pressure \u2026 Settles for contested jumpshots from outside and takes a lot of fading and offbalance attempts inside \u2026 Leaves his feet to make passes and gets caught with no options \u2026 Is stuck between positions because he has not proven that he can be a consistent scorer or that his decision making will translate to the next level \u2026", "Not a true point, more of a combo \u2026 Must continue to develop his point guard skills \u2026 Can learn to become a better floor general, distributing the ball, setting everyone up \u2026 Needs to gain experience \u2026 Must get stronger physically, to be able to finish off drives as well as shoot with more consistency \u2026 His ball handling ability is solid, but can improve \u2026"

"""


In [2]:
summarizer = pipeline("text-generation", model="Qwen/Qwen3-4B-Instruct-2507")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [4]:
def summarize_text(text_blob):
    messages = [{"role": "user", "content": "Summarize the following text as a list of bullet points. 10 bullet points maximum. Don't include any other dialog, just the bullet points:\n\n%s" % text_blob}]
    out = summarizer(messages)
    return out[0]["generated_text"][-1]["content"]

In [6]:
adem = """Long, athletic shot blocking big who has potential as a defensive anchor … Good timing on the shot block … Very long and has a quick jump …7-3 wingspan combined with a 40 inch max vert are elite numbers … Isn’t the most fluid looking running, but Bona is very fast in a straight line in the open court and does run the break … Ferocious dunker … Has good hands to catch a lob or a bounce pass on a pick and roll into a slam … Aggressive with the ball in his hands … Showed solid improvement in free throw shooting at UCLA (57% as a freshman to 70% last season) … Decent hands around the hoop overall … Shows some semblance of a post game and a quick, fluid, and powerful drop step into a baby hook shot, but likely will need to develop further polish … Sets a decent screen, but sometimes gets impatient and leaves for the role a bit early … Had one of the best standing reach and wingspan measurements at the Draft Combine, especially considering his height …
"""

In [7]:
print(summarize_text(adem))

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


- Long, athletic shot blocker with potential as a defensive anchor  
- Excellent timing and quick jump on shot blocks  
- 7-3 wingspan combined with a 40-inch maximum vertical jump (elite numbers)  
- Fast in a straight line on open court, though not the most fluid runner  
- Ferocious dunker with strong hands for catching lobs or bounce passes  
- Aggressive play with the ball in his hands  
- Improved free throw shooting from 57% (freshman year) to 70% (last season)  
- Decent hands around the hoop; shows potential for a post game  
- Has a quick, fluid, and powerful drop step into a baby hook shot (needs further development)  
- Sets decent screens but sometimes leaves early due to impatience


In [8]:
player_df = pd.read_json("/kaggle/input/datasets/caseydurfee/player-description-df/player_descriptions.json")

In [5]:
player_df.iloc[0]

nbadraft_net_comps                                     [Kenneth Faried]
descr_other           [Overall: Considered a high character guy … Go...
descr_strengths       [A very athletic and versatile PF … Is a very ...
descr_weaknesses      [Gordon’s main weaknesses revolve around his l...
descr_raw             <div class="vc_tta-panel vc_active" data-vc-co...
Athleticism                                                         9.0
Size                                                                8.0
Defense                                                             8.0
Strength                                                            7.0
Quickness                                                           8.0
Leadership                                                          8.0
Jump Shot                                                           6.0
NBA Ready                                                           8.0
Rebounding                                                      

In [9]:
import time
import bs4 as bs

In [ ]:
#df2 = pd.DataFrame(columns=['player_name', 'summarized'])

df2 = pd.read_json("player_summaries.json")

for idx, row in player_df.iterrows():
    # skip row if done before.
    if sum(df2.player_name == row.Name) > 0:
        continue
    dr = row.descr_raw

    soup = bs.BeautifulSoup(dr, 'html.parser')

    summary_chunks = []
    _start = time.time()
    for p in soup.find(class_='vc_tta-panel-body').find_all('p'):
        if len(p.text) > 100:
            summary_chunks.append(p.text)

    all_text = "\n".join(summary_chunks)
    if len(all_text) < 100:
        summary = ""
    else:
        summary = summarize_text(all_text)
    print(f"took {time.time() - _start} for {row.Name}")
    df2.loc[len(df2)] = [row.Name, summary]
    print(summary)
    # save after each player, because paranoia
    df2.to_json("player_summaries.json")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


took 16.428269386291504 for Aaron Gordon
- Athletic and versatile power forward with exceptional leaping ability, explosiveness, and high motor  
- Physically lean but wiry and strong, confident in contact and effective in rebounding (especially offensive)  
- Excellent ball awareness, quick decision-making, and strong passing for a PF (2 apg as a freshman)  
- Moves efficiently without the ball, creates easy scoring opportunities near the rim, and is an alley-oop magnet  
- Strong defensive presence: contests shots, disciplined in closeouts, understands help-side defense, and can play multiple positions  
- High IQ, focused, well-conditioned, and rarely fatigues—acts as a reliable role player and team glue  
- Makes strong finishes and has a solid first step to create shots on drives to his right  
- Shows good defensive fundamentals, including shot-blocking and effort, despite limited size  
- Needs to improve offensive skills, particularly shooting (especially from beyond the arc an

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


took 13.483250856399536 for Aaron Holiday
- Quick guard with excellent speed and ability to penetrate opponents off the dribble  
- Tough to defend due to strong ball handling and court awareness  
- Exceptional court vision, quick decision-making, and strong sense of when to score or pass  
- Thrives in transition, excelling as a playmaker and quick decision-maker  
- Smooth shooting form, 42.9% from three-point range and 82% from the free-throw line  
- Scores efficiently from all levels: around the rim, mid-range, and three-point range  
- Selfless, makes smart passes, and frequently kicks out to open shooters  
- Improved significantly from sophomore to junior year (12.3 to 20.3 ppg)  
- Has NBA pedigree through older brothers Jrue and Justin Holiday, who played in the league  
- Listed at 6’1”, 185 lbs with a 6’6 wingspan; small for an NBA point guard but has good length and defensive awareness


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

took 19.218239307403564 for Cam Whitmore
- 6'6", 230 lb swingman with elite athleticism, explosive 2-footed leaping ability, and strong body control  
- Excels as a downhill driver, applying pressure and finishing efficiently with 12.5 ppg and 58% FG inside the arc  
- Strong perimeter finisher with finesse, able to navigate congested paint areas despite physical build  
- Effective off-ball cutter who finds open looks and contributes in transition and spacing  
- Solid rebounder (5.3 rpg) and active defender with good shot-blocking (1.4 spg) and aggressive defensive instincts  
- Patient with the ball, uses hesitation moves and slashing to create scoring opportunities  
- Willing to match up with bigger players, shows versatility and positionless basketball mentality  
- Reliable spot-up shooter (34% 3PT) and can create space with jab steps and footwork  
- Physical tools and energy allow him to play SG-PF at the pro level with experience and development  
- Inexperience evident in sh

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


took 16.109877109527588 for Cameron Bairstow
- Strong mid-range jumper knockdown ability and physical presence  
- Hustles hard, especially in the low post and on the boards  
- Will sacrifice his body for team success and applies pressure on opponents  
- Consistently reaches the free throw line (8.8 FTA in 2013–14, 4th nationally)  
- Excellent motor, strong rebounding, and effective post positioning  
- Dominates offensively in the paint; capable of shooting elbow and baseline jumpers  
- One of the nation’s most improved players (2012–13 to 2013–14) with shot-blocking ability as a senior  
- Has elite power forward measurables, comparable to Julius Randle  
- High shooting percentage (74% FT, 56% FG senior season)  

- Turned 24 during the season, reducing long-term upside  
- Below-average athlete by NBA standards; limited below-the-rim ability  
- College success may not translate to professional level due to physical differences  
- Relatively inexperienced, with limited signifi

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


took 12.272838830947876 for Cameron Johnson
- Tall, wiry forward with a smooth shooting motion and difficult-to-contest release  
- Consistently scores off ball-screens, dribble pull-ups, and heavily guarded shots  
- Excellent range and above-average catch-and-shoot ability  
- High shooting efficiency: 16.9 points per game on 11.6 field goal attempts and 46% from three  
- Shot 57% from inside the arc due to improved mid-range game  
- Impressive movement without the ball, actively cuts off screens to create open looks  
- Elite free-throw shooter (80%+ over three seasons as a starter)  
- Improved defensive rating and mobility after offseason hip surgery  
- Averages 1.2 steals per game and is an active defender despite limited athleticism  
- Outlook: Mid-second round to late first round draft pick; strong potential as a three-and-D player if defensive growth continues
